In [ ]:
import os
from typing import TypedDict
from dotenv import load_dotenv 

load_dotenv() 

from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import StateGraph, START, END
from langchain_community.tools.tavily_search import TavilySearchResults

os.environ["LANGCHAIN_TRACING_V2"] = "false"
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.7
)

search_tool = TavilySearchResults(max_results=3)

class BlogState(TypedDict):
    topic: str
    plan: str
    research_content: str
    draft: str
    critique: str
    revision_count: int
    max_revisions: int

def planner_node(state: BlogState) -> dict:
    print("\n---  PLANNER NODE (Gemini) ---")
    topic = state["topic"]
    
    prompt = f"""
    You are an expert content strategist. Create a detailed outline/plan for a high-quality blog post on the following topic:
    Topic: {topic}
    
    The plan should include:
    1. Catchy Title Ideas
    2. Target Audience
    3. Key Sections and Subheadings
    4. Key questions or research points to cover in each section.
    """
    
    response = llm.invoke(prompt)
    return {"plan": response.content}

def researcher_node(state: BlogState) -> dict:
    print("\n--- RESEARCHER NODE ---")
    topic = state["topic"]
    
    try:
        search_query = f"{topic} key facts and latest developments"
        results = search_tool.invoke({"query": search_query})
        research_summary = "\n".join([r.get("content", "") for r in results])
        print(f"Fetched {len(results)} search results.")
    except Exception as e:
        print("Search failed, proceeding with model knowledge.")
        research_summary = "No live web results retrieved."
        
    return {"research_content": research_summary}

def writer_node(state: BlogState) -> dict:
    print("\n---  WRITER NODE (Gemini) ---")
    topic = state["topic"]
    plan = state["plan"]
    research = state["research_content"]
    critique = state.get("critique", "")
    
    if critique:
        print("Revising draft based on critique...")
        prompt = f"""
        You are a professional technical blog writer.
        Revise the previous blog draft based on the following critique:
        
        Original Topic: {topic}
        Outline/Plan: {plan}
        Research Data: {research}
        
        CRITIQUE / FEEDBACK:
        {critique}
        
        Write an improved, publication-ready blog post in Markdown format.
        """
    else:
        print("Writing first draft...")
        prompt = f"""
        You are a professional blog writer. Write an engaging, well-structured blog post in Markdown based on:
        
        Topic: {topic}
        Outline/Plan: {plan}
        Research Data: {research}
        
        Requirements:
        - Use clean Markdown formatting (Headers, Bullet points).
        - Engaging intro and concluding summary.
        """
    
    response = llm.invoke(prompt)
    return {"draft": response.content}

def critic_node(state: BlogState) -> dict:
    print("\n--- CRITIC NODE (Gemini) ---")
    draft = state["draft"]
    topic = state["topic"]
    
    prompt = f"""
    You are a strict editorial reviewer. Review the following blog draft on '{topic}':
    
    DRAFT:
    {draft}
    
    Evaluate it on:
    1. Clarity and Structure
    2. Tone and Engagement
    3. Accuracy and Detail
    
    If it is excellent and ready to publish, reply ONLY with the single word 'APPROVED'.
    Otherwise, provide short, actionable feedback/critique on what needs improvement.
    """
    
    response = llm.invoke(prompt)
    critique = response.content.strip()
    
    current_revisions = state.get("revision_count", 0) + 1
    return {
        "critique": critique,
        "revision_count": current_revisions
    }
def should_continue(state: BlogState) -> str:
    critique = state["critique"]
    revision_count = state["revision_count"]
    max_revisions = state["max_revisions"]
    
    if "APPROVED" in critique.upper() or revision_count >= max_revisions:
        print(f"\n Workflow Complete! Status: {'APPROVED' if 'APPROVED' in critique.upper() else 'Max Revisions Limit Reached'}")
        return END
    else:
        print(f"\n Revision Required ({revision_count}/{max_revisions}). Sending back to Writer...")
        return "writer"

workflow = StateGraph(BlogState)

workflow.add_node("planner", planner_node)
workflow.add_node("researcher", researcher_node)
workflow.add_node("writer", writer_node)
workflow.add_node("critic", critic_node)

workflow.add_edge(START, "planner")
workflow.add_edge("planner", "researcher")
workflow.add_edge("researcher", "writer")
workflow.add_edge("writer", "critic")

workflow.add_conditional_edges("critic", should_continue, {
    "writer": "writer",
    END: END
})

app = workflow.compile()

if __name__ == "__main__":
    initial_input = {
        "topic": "Agentic AI and LangGraph for Autonomous Workflows",
        "revision_count": 0,
        "max_revisions": 2
    }
    
    print(" Starting LangGraph Agent with Google AI Studio API...")
    final_state = app.invoke(initial_input)
    
  
    file_name = "final_blog_post.md"
    with open(file_name, "w", encoding="utf-8") as f:
        f.write(final_state["draft"])
        
    print("\n" + "="*50)
    print(f" SUCCESS! ")
    print(f" your file '{file_name}' has been saved.")
    print("="*50 + "\n")



C:\Users\user\AppData\Local\Temp\ipykernel_12456\1565006608.py:9: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools.tavily_search import TavilySearchResults
C:\Users\user\AppData\Local\Temp\ipykernel_12456\1565006608.py:17: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  search_tool = TavilySearchResults(max_results=3)


🚀 Starting LangGraph Agent with Google AI Studio API...

--- PLANNER NODE (Gemini) ---

--- RESEARCHER NODE ---
Fetched 3 search results.

--- WRITER NODE (Gemini) ---
Writing first draft...

--- CRITIC NODE (Gemini) ---

 Workflow Complete! Status: APPROVED

 SUCCESS! 
 Aapki file 'final_blog_post.md' has been saved.

